# Spark SQL Window Functions

## Basic to intermediate SQL patterns

Window functions calculate values from related rows while preserving every input row. This SQL companion to `D320_SparkWindowFunctions.ipynb` covers ranking, distribution, offset, value, and aggregate windows using Spark SQL.

## Learning objectives

By the end of this lesson, you should be able to:

- Write an `OVER` clause with `PARTITION BY`, `ORDER BY`, and a frame.
- Use all core ranking and distribution window functions.
- Compare nearby rows with `LAG` and `LEAD`.
- Retrieve ordered values with `FIRST_VALUE`, `LAST_VALUE`, and `NTH_VALUE`.
- Build whole-partition, cumulative, rolling, and value-range aggregates.
- Solve top-N and deduplication problems with CTEs.
- Identify common correctness and performance issues.

## 1. Start Spark before running the notebook

This notebook follows the current `D30_SparkDF` setup: a single-node Spark **standalone cluster**. It does not use the older source notebook's `findspark` or `local[4]` configuration.

```bash
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh spark://$(hostname):7077
```

Set `SPARK_MASTER` if the master URL differs from the hostname-based default.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D32-Spark-SQL-Window-Functions")
    .master(master_url)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

## 2. Create the employee view

Spark SQL queries tables and views. We first create a DataFrame with an explicit schema, convert the date text, and register it as the temporary view `employees`. The sample deliberately contains salary ties and null bonuses.

In [ ]:
employee_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("employee_name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False),
    StructField("bonus", IntegerType(), True),
    StructField("join_date", StringType(), False),
])

employee_rows = [
    (101, "Anika",  "Engineering", 90000, 9000,  "2022-01-10"),
    (102, "Bala",   "Engineering", 78000, None,  "2023-03-15"),
    (103, "Chen",   "Engineering", 78000, 6000,  "2023-07-01"),
    (104, "Divya",  "Engineering", 65000, 4000,  "2024-02-20"),
    (201, "Farah",  "Finance",     85000, 8000,  "2021-11-05"),
    (202, "Venkat",  "Finance",     72000, 5000,  "2023-01-12"),
    (203, "Harini", "Finance",     72000, None,  "2023-09-18"),
    (204, "Ivan",   "Finance",     60000, 3500,  "2024-05-22"),
    (301, "Jaya",   "Sales",       82000, 12000, "2022-06-30"),
    (302, "Kiran",  "Sales",       70000, 10000, "2023-04-08"),
    (303, "Leela",  "Sales",       70000, 7000,  "2023-12-11"),
    (304, "Manoj",  "Sales",       58000, None,  "2024-06-17"),
]

employees = (
    spark.createDataFrame(employee_rows, employee_schema)
    .withColumn("join_date", F.to_date("join_date"))
)
employees.createOrReplaceTempView("employees")
spark.sql("SELECT * FROM employees ORDER BY department, salary DESC, employee_id").show(truncate=False)

## 3. SQL window anatomy

A window function is followed by an `OVER` clause:

```sql
function(...) OVER (
    PARTITION BY grouping_columns
    ORDER BY ordering_columns
    ROWS BETWEEN frame_start AND frame_end
)
```

- `PARTITION BY` creates independent analytical groups.
- `ORDER BY` defines sequence within each group.
- The optional frame identifies the rows visible to frame-sensitive functions.

Unlike `GROUP BY`, a window does not reduce each group to one row.

## 4. `ROW_NUMBER`: a unique sequence

`ROW_NUMBER` assigns `1, 2, 3, ...` within each department. Salary alone does not determine the order of tied rows, so `employee_id` provides a stable tie-breaker.

In [ ]:
spark.sql("""
SELECT
    employee_id, employee_name, department, salary,
    ROW_NUMBER() OVER (
        PARTITION BY department
        ORDER BY salary DESC, employee_id
    ) AS row_number
FROM employees
ORDER BY department, row_number
""").show()

## 5. `RANK` and `DENSE_RANK`: ties

Both give equal salaries the same position. `RANK` leaves a gap after a tie (`1, 2, 2, 4`); `DENSE_RANK` does not (`1, 2, 2, 3`). Do not add `employee_id` to these window orderings when equal salaries should remain tied.

In [ ]:
spark.sql("""
SELECT
    employee_id, employee_name, department, salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS salary_rank,
    DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dense_salary_rank
FROM employees
ORDER BY department, salary_rank, employee_id
""").show()

## 6. `PERCENT_RANK`: position based on rank

For four descending scores `100, 90, 90, 70`, `PERCENT_RANK` uses:

```text
(rank - 1) / (number of rows - 1)
```

| Score | Rank | Calculation | percent_rank |
| ---: | ---: | --- | ---: |
| 100 | 1 | (1 - 1) / 3 | 0.000 |
| 90 | 2 | (2 - 1) / 3 | 0.333 |
| 90 | 2 | (2 - 1) / 3 | 0.333 |
| 70 | 4 | (4 - 1) / 3 | 1.000 |

Tied rows share the same rank and percentage. The first row is always `0.0`; Spark returns `0.0` for a one-row partition too.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW scores(score) AS VALUES (100), (90), (90), (70)")

spark.sql("""
SELECT
    score,
    RANK() OVER (ORDER BY score DESC) AS score_rank,
    ROUND(PERCENT_RANK() OVER (ORDER BY score DESC), 3) AS percent_rank
FROM scores
ORDER BY score DESC
""").show()

## 7. `CUME_DIST`: cumulative proportion

For the same descending scores, `CUME_DIST` is the number of rows at or before the current ordered value divided by the total row count. In descending order, that means scores greater than or equal to the current score.

| Score | Rows at or before it | Calculation | cume_dist |
| ---: | ---: | --- | ---: |
| 100 | 1 | 1 / 4 | 0.25 |
| 90 | 3 | 3 / 4 | 0.75 |
| 90 | 3 | 3 / 4 | 0.75 |
| 70 | 4 | 4 / 4 | 1.00 |

The cumulative position moves to the end of the complete tie group, so both 90s receive `0.75`.

In [ ]:
spark.sql("""
SELECT
    score,
    ROUND(CUME_DIST() OVER (ORDER BY score DESC), 2) AS cume_dist
FROM scores
ORDER BY score DESC
""").show()

## 8. `NTILE`: distribute rows into buckets

`NTILE(n)` places ordered rows into `n` numbered buckets whose sizes differ by at most one. A deterministic ordering makes the bucket assignment reproducible when salaries tie.

In [ ]:
spark.sql("""
SELECT
    employee_id, employee_name, department, salary,
    NTILE(4) OVER (
        PARTITION BY department
        ORDER BY salary DESC, employee_id
    ) AS salary_quartile
FROM employees
ORDER BY department, salary_quartile, salary DESC
""").show()

## 9. `LAG` and `LEAD`: compare nearby rows

`LAG` reads an earlier row and `LEAD` reads a later row in the window order. Their second argument is the offset; the optional third argument is returned when no row exists at that offset.

In [ ]:
spark.sql("""
WITH comparisons AS (
    SELECT
        employee_id, employee_name, department, salary,
        LAG(salary, 1) OVER (
            PARTITION BY department ORDER BY salary DESC, employee_id
        ) AS previous_salary,
        LEAD(salary, 1) OVER (
            PARTITION BY department ORDER BY salary DESC, employee_id
        ) AS next_salary
    FROM employees
)
SELECT *, salary - previous_salary AS difference_from_previous
FROM comparisons
ORDER BY department, salary DESC, employee_id
""").show()

## 10. `FIRST_VALUE`, `LAST_VALUE`, and `NTH_VALUE`

These functions retrieve a value from an ordered window. With an ordered window, the default frame usually ends at the current row. Define the whole-partition frame explicitly when `LAST_VALUE` should mean the final value in the department. `NTH_VALUE` uses a 1-based position.

In [ ]:
spark.sql("""
SELECT
    employee_id, employee_name, department, salary,
    FIRST_VALUE(employee_name) OVER salary_window AS highest_paid_employee,
    LAST_VALUE(employee_name) OVER salary_window AS lowest_paid_employee,
    NTH_VALUE(salary, 2) OVER salary_window AS second_ordered_salary
FROM employees
WINDOW salary_window AS (
    PARTITION BY department
    ORDER BY salary DESC, employee_id
    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
)
ORDER BY department, salary DESC, employee_id
""").show()

### Ignore nulls in value functions

Spark SQL supports `IGNORE NULLS` with `FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`, `LAG`, and `LEAD`. The following query returns the earliest non-null bonus in each department.

In [ ]:
spark.sql("""
SELECT
    employee_name, department, bonus, join_date,
    FIRST_VALUE(bonus) IGNORE NULLS OVER (
        PARTITION BY department
        ORDER BY join_date, employee_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS first_recorded_bonus
FROM employees
ORDER BY department, join_date
""").show()

## 11. Aggregate functions over a partition

Standard aggregates become window functions when followed by `OVER (...)`. With only `PARTITION BY`, every employee row sees statistics calculated from the entire department.

In [ ]:
spark.sql("""
SELECT
    employee_name, department, salary,
    COUNT(*) OVER (PARTITION BY department) AS department_size,
    SUM(salary) OVER (PARTITION BY department) AS department_payroll,
    ROUND(AVG(salary) OVER (PARTITION BY department), 2) AS department_avg_salary,
    MIN(salary) OVER (PARTITION BY department) AS department_min_salary,
    MAX(salary) OVER (PARTITION BY department) AS department_max_salary,
    ROUND(salary - AVG(salary) OVER (PARTITION BY department), 2)
        AS salary_minus_department_avg
FROM employees
ORDER BY department, salary DESC
""").show()

## 12. Row frames: cumulative and rolling calculations

A `ROWS` frame selects physical row positions relative to the current row.

- `UNBOUNDED PRECEDING ... CURRENT ROW` creates a running calculation.
- `1 PRECEDING ... CURRENT ROW` includes the current row and one previous row.

In [ ]:
spark.sql("""
SELECT
    employee_name, department, salary, join_date,
    SUM(salary) OVER (
        PARTITION BY department ORDER BY join_date, employee_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_payroll,
    ROUND(AVG(salary) OVER (
        PARTITION BY department ORDER BY join_date, employee_id
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ), 2) AS two_employee_moving_avg
FROM employees
ORDER BY department, join_date, employee_id
""").show()

## 13. Range frames: calculations by ordered value

A `RANGE` frame compares ordering values rather than counting rows. With salary as the single numeric ordering expression, `10000 PRECEDING` through `CURRENT ROW` includes salaries from 10,000 below the current salary through the current salary. Equal salary peers are included together.

In [ ]:
spark.sql("""
SELECT
    employee_name, department, salary,
    COUNT(*) OVER (
        PARTITION BY department ORDER BY salary
        RANGE BETWEEN 10000 PRECEDING AND CURRENT ROW
    ) AS employees_in_10k_band,
    ROUND(AVG(salary) OVER (
        PARTITION BY department ORDER BY salary
        RANGE BETWEEN 10000 PRECEDING AND CURRENT ROW
    ), 2) AS average_in_10k_band
FROM employees
ORDER BY department, salary, employee_id
""").show()

## 14. Top N per group with a CTE

A window result cannot normally be referenced directly in the same query's `WHERE` clause because filtering occurs before window evaluation. Calculate the position in a CTE, then filter in the outer query. Use `ROW_NUMBER` for exactly N rows or `DENSE_RANK` to retain all ties at the Nth salary level.

In [ ]:
spark.sql("""
WITH ranked_employees AS (
    SELECT
        employee_id, employee_name, department, salary,
        ROW_NUMBER() OVER (
            PARTITION BY department ORDER BY salary DESC, employee_id
        ) AS position
    FROM employees
)
SELECT *
FROM ranked_employees
WHERE position <= 2
ORDER BY department, position
""").show()

## 15. Contribution to group total

Divide a row value by a windowed group total to calculate its percentage contribution.

In [ ]:
spark.sql("""
SELECT
    employee_name, department, salary,
    SUM(salary) OVER (PARTITION BY department) AS department_payroll,
    ROUND(
        salary * 100.0 / SUM(salary) OVER (PARTITION BY department),
        2
    ) AS payroll_share_pct
FROM employees
ORDER BY department, salary DESC
""").show()

## 16. Deduplicate and retain the latest record

Partition by the business key, order newest first, assign `ROW_NUMBER`, and retain row 1. Add a stable tie-breaker when timestamps may be identical.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW status_updates(employee_id, status, updated_at, event_id) AS VALUES
    (101, 'ACTIVE',   TIMESTAMP '2026-08-25 09:00:00', 1),
    (101, 'ON_LEAVE', TIMESTAMP '2026-08-27 14:30:00', 2),
    (202, 'ACTIVE',   TIMESTAMP '2026-08-26 10:15:00', 1),
    (202, 'INACTIVE', TIMESTAMP '2026-08-28 08:00:00', 2)
""")

spark.sql("""
WITH latest AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY employee_id
            ORDER BY updated_at DESC, event_id DESC
        ) AS row_number
    FROM status_updates
)
SELECT employee_id, status, updated_at, event_id
FROM latest
WHERE row_number = 1
ORDER BY employee_id
""").show(truncate=False)

## 17. Find the second distinct salary

`DENSE_RANK` directly expresses an Nth distinct value. This is clearer than nesting `DISTINCT`, `ROW_NUMBER`, and multiple CTEs.

In [ ]:
spark.sql("""
WITH salary_levels AS (
    SELECT
        employee_id, employee_name, department, salary,
        DENSE_RANK() OVER (ORDER BY salary DESC) AS salary_level
    FROM employees
)
SELECT employee_name, department, salary
FROM salary_levels
WHERE salary_level = 2
ORDER BY employee_id
""").show()

## 18. Inspect the execution plan

Partitioned windows generally require an exchange to bring equal keys together and a sort for the window order. Use `EXPLAIN FORMATTED` to inspect these operations.

In [ ]:
plan_rows = spark.sql("""
EXPLAIN FORMATTED
SELECT
    employee_name, department, salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS salary_rank
FROM employees
""").collect()
print("\n".join(row[0] for row in plan_rows))

## 19. Correctness and performance checklist

- Add a tie-breaker for deterministic `ROW_NUMBER`, `LAG`, `LEAD`, and positional value results.
- Do not add a tie-breaker to `RANK` or `DENSE_RANK` when business values should remain tied.
- Define the frame explicitly for `LAST_VALUE`, running totals, and moving calculations.
- Remember that `ROWS` counts positions while `RANGE` compares ordered values.
- Avoid omitting `PARTITION BY` on large data unless a single global window is required.
- Expect shuffling and sorting, and check for skewed partition keys.
- Use a CTE or subquery to filter by a window result.
- Filtering before the window changes which rows the calculation can see.
- `COUNT(*)` counts rows; `COUNT(nullable_column)` counts only non-null values.

## 20. Practice exercises

1. Return exactly the three lowest-paid employees in each department.
2. Calculate each employee's difference from the next lower salary.
3. Create a three-row moving salary average ordered by join date.
4. Return all employees in the two highest distinct salary levels per department.
5. Show the earliest and latest joiner for each department on every row.

In [ ]:
# Write the practice SQL inside spark.sql("""...""") calls.

## Summary

Spark SQL window functions preserve detail rows while adding calculations derived from related rows. Choose the business partition, define a meaningful order, and specify the frame when row visibility matters. CTEs make window-result filtering clear and support common data-engineering patterns such as top-N selection and deduplication.

## Stop Spark

Run this cell when the lesson is complete.

In [ ]:
spark.stop()
print("Spark session stopped.")